In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!gmx --version

!apt-get remove --purge gromacs


Mounted at /content/drive
/bin/bash: line 1: gmx: command not found
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'gromacs' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
# Download the latest version of GROMACS 2024.3
!wget https://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz

# Extract the tarball
!tar xfz gromacs-2024.4.tar.gz
# Navigate to the GROMACS source directory
%cd gromacs-2024.4

# Create a build directory
!mkdir build
%cd build

# Configure the build with CUDA (GPU) support
!cmake .. -DGMX_BUILD_OWN_FFTW=ON -DGMX_GPU=CUDA -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs

# Compile the source code using all available cores
!make -j$(nproc)

# Install GROMACS
!make install


--2025-12-14 23:10:42--  https://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz
Resolving ftp.gromacs.org (ftp.gromacs.org)... 130.237.11.165, 2001:6b0:1:1191:216:3eff:fec7:6e30
Connecting to ftp.gromacs.org (ftp.gromacs.org)|130.237.11.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42356162 (40M) [application/x-gzip]
Saving to: ‘gromacs-2024.4.tar.gz’

gromacs-2024.4.tar. 100%[===================>]  40.39M  12.8MB/s    in 3.2s    

2025-12-14 23:10:46 (12.8 MB/s) - ‘gromacs-2024.4.tar.gz’ saved [42356162/42356162]

/content/gromacs-2024.4
/content/gromacs-2024.4/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working C

In [ ]:
# Manually export GROMACS binary to the PATH
import os
os.environ['PATH'] += ":/usr/local/gromacs/bin"



In [ ]:
!gmx --version

                         :-) GROMACS - gmx, 2024.4 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /content/gromacs-2024.4/build
Command line:
  gmx --version

GROMACS version:     2024.4
Precision:           mixed
Memory model:        64 bit
MPI library:         thread_mpi
OpenMP support:      enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:         CUDA
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8
SIMD instructions:   AVX_512
CPU FFT library:     fftw-3.3.8-sse2-avx-avx2-avx2_128-avx512
GPU FFT library:     cuFFT
Multi-GPU FFT:       none
RDTSCP usage:        enabled
TNG support:         enabled
Hwloc support:       disabled
Tracing support:     disabled
C compiler:          /usr/bin/cc GNU 11.4.0
C compiler flags:    -fexcess-precision=fast -funroll-all-loops -march=skylake-avx512 -Wno-missing-field-initializers -O3 -DNDEBUG
C++ compiler:        /usr/bin/c++ GNU 11.4.0
C++ compiler flags:  -fexcess-precision=fast -funroll-all-l

In [ ]:
# Install required packages
!apt-get install -y csh
!apt-get update
!apt-get install -y build-essential cmake git libfftw3-dev libgsl-dev

# Install CUDA (if not already installed)
!apt-get install cuda

# Check if GPU is available (optional, but useful for verification)
!nvidia-smi

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  csh
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 245 kB of archives.
After this operation, 358 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 csh amd64 20110502-7 [245 kB]
Fetched 245 kB in 1s (229 kB/s)
Selecting previously unselected package csh.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../csh_20110502-7_amd64.deb ...
Unpacking csh (20110502-7) ...
Setting up csh (20110502-7) ...
update-alternatives: using /bin/bsd-csh to provide /bin/csh (csh) in auto mode
Processing triggers for man-db (2.10.2-1) ...
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/l

In [ ]:
import sys
import os

def incorporate_silicon(input_pdb, output_pdb):
    """
    Reads a PDB file and replaces every Alpha Carbon (CA) with Silicon (SI).
    """
    with open(input_pdb, 'r') as file:
        lines = file.readlines()

    modified_lines = []
    si_count = 0

    print(f"Processing {input_pdb}...")

    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            # Extract Atom Name (Columns 13-16)
            atom_name = line[12:16]

            # Check if the atom is an Alpha Carbon (CA)
            if atom_name.strip() == "CA":
                # 1. Replace Atom Name 'CA' with 'SI'
                # We interpret "Silicon Incorporated" as replacing the backbone carbon.
                # PDB columns are fixed-width. Atom name is cols 13-16.
                # We use " SI " to maintain alignment.
                line_prefix = line[:12]
                line_suffix = line[16:]

                # Construct new line with SI atom name
                # " SI " puts 'S' in col 14 and 'I' in col 15, standard for 2-letter elements
                new_atom_name = " SI "

                # Update the line
                line = line_prefix + new_atom_name + line_suffix

                # 2. Update Element Symbol (Columns 77-78) to 'Si'
                # PDB standard places element symbol at the end of the line
                if len(line) >= 78:
                    line = line[:76] + "Si" + line[78:]
                else:
                    # Pad line if it's too short (rare but possible in messy PDBs)
                    line = line.strip().ljust(76) + "Si" + "\n"

                si_count += 1

        modified_lines.append(line)

    # Write the modified file
    with open(output_pdb, 'w') as file:
        file.writelines(modified_lines)

    print(f"Done! Generated {output_pdb}")
    print(f"Total residues modified (CA -> Si): {si_count}")

# --- Execution Block ---
if __name__ == "__main__":
    # Input filename from your upload
    input_filename = "AF-E7EUV6-F1-model_v4.pdb"
    output_filename = "e7euv6_si.pdb"

    if os.path.exists(input_filename):
        incorporate_silicon(input_filename, output_filename)
    else:
        print(f"Error: Could not find {input_filename} in the current directory.")
        # Fallback for testing if you paste the content into a file named 'protein.pdb'
        # incorporate_silicon('protein.pdb', 'protein_si.pdb')

/content/drive/MyDrive/MD/Silica
Processing AF-E7EUV6-F1-model_v4.pdb...
Done! Generated e7euv6_si.pdb
Total residues modified (CA -> Si): 106


In [ ]:
import os
import shutil

def build_silicon_forcefield():
    # 1. Create directory structure
    ff_name = "silicon.ff"
    if os.path.exists(ff_name):
        shutil.rmtree(ff_name)
    os.makedirs(ff_name)

    print(f"Building custom forcefield: {ff_name}...")

    # We need base files. In a real scenario, we copy from $GMXLIB.
    # Here, we will generate minimal valid AMBER-based files for the patch.
    # NOTE: For this code to run robustly on your machine, it assumes standard
    # AMBER99SB files exist. If not, we will create simplified versions below.

    # --- A. RTP File (Residue Topologies) ---
    # We will read the system's amber99sb.ff/aminoacids.rtp if available,
    # or write a patch. For robustness, here is a patcher logic.

    print("Instructions: This script requires a base force field to copy.")
    print("Please ensure GROMACS is installed. We will attempt to use 'amber99sb.ff'.")

    # Try to locate GROMACS library path (heuristic)
    gmx_lib = "//usr/local/gromacs/top" # Common Linux path
    if "GMXLIB" in os.environ:
        gmx_lib = os.environ["GMXLIB"]

    base_ff = os.path.join(gmx_lib, "amber99sb.ff")

    # If standard path fails, use local directory assuming user copied it
    if not os.path.exists(base_ff):
        print(f"WARNING: Could not find {base_ff}.")
        print("Please copy your system's 'amber99sb.ff' folder to this directory and rename it 'base.ff'")
        print("For now, I will generate the configuration files assuming 'base.ff' exists.")
        base_ff = "base.ff"

    if not os.path.exists(base_ff):
        print("Error: Base force field not found. Cannot patch.")
        return

    # Copy all files
    for item in os.listdir(base_ff):
        s = os.path.join(base_ff, item)
        d = os.path.join(ff_name, item)
        if os.path.isfile(s):
            shutil.copy2(s, d)

    # --- Modify aminoacids.rtp ---
    # We replace atom 'CA' (type CT) with atom 'SI' (type SZ)
    rtp_path = os.path.join(ff_name, "aminoacids.rtp")
    with open(rtp_path, 'r') as f:
        rtp_data = f.read()

    # Replace atom definitions
    # " CA   CT "  -> " SI   SZ "
    rtp_data = rtp_data.replace(" CA   CT", " SI   SZ")
    rtp_data = rtp_data.replace(" CA   CX", " SI   SZ") # For N-terminal variants

    # Replace bond definitions
    rtp_data = rtp_data.replace(" CA ", " SI ")
    rtp_data = rtp_data.replace(" CA+", " SI+")
    rtp_data = rtp_data.replace("-CA", "-SI")

    with open(rtp_path, 'w') as f:
        f.write(rtp_data)

    # --- Modify atomtypes.atp ---
    # Add Silicon atom type
    atp_path = os.path.join(ff_name, "atomtypes.atp")
    with open(atp_path, 'a') as f:
        # Type  Mass
        f.write("\nSZ      28.08550 ; Silicon Backbone\n")

    # --- Modify ffnonbonded.itp ---
    # Define LJ parameters for SZ (Silicon).
    # We borrow CT (Carbon) params but can slightly increase sigma if desired.
    # CT params: sigma=0.33996, epsilon=0.45773 (Amber99SB approx)
    nb_path = os.path.join(ff_name, "ffnonbonded.itp")
    with open(nb_path, 'r') as f:
        lines = f.readlines()

    with open(nb_path, 'w') as f:
        for line in lines:
            f.write(line)
            # Find the CT entry to clone
            if line.strip().startswith("CT") and "6" in line and "12" not in line:
                # Clone CT parameters for SZ
                parts = line.split()
                # Assuming format: type atnum mass charge ptype sigma epsilon
                new_line = line.replace("CT", "SZ", 1)
                new_line = new_line.replace("6", "14", 1) # Change atomic number 6->14 (Si)
                new_line = new_line.replace("12.01", "28.08", 1) # Mass
                f.write(new_line)

    # --- Modify ffbonded.itp ---
    # Duplicate all bonded interactions involving CT to apply to SZ
    bond_path = os.path.join(ff_name, "ffbonded.itp")
    with open(bond_path, 'r') as f:
        bond_lines = f.readlines()

    with open(bond_path, 'w') as f:
        for line in bond_lines:
            f.write(line)
            # If line contains CT, duplicate it for SZ
            if "CT" in line and not line.startswith(";"):
                # Simple permutations: CT->SZ
                # Note: This covers CT-CT, CT-N, etc.
                # Ideally we do permutations (CT-SZ, SZ-SZ), but for a pure backbone replacement:
                new_line = line.replace("CT", "SZ")

In [ ]:
import os
import shutil
import subprocess
import re
import itertools
from collections import defaultdict

# --- CONFIGURATION ---
WORKING_DIR = "/content/drive/MyDrive/MD/Silica/"
INPUT_PDB_NAME = "AF-E7EUV6-F1-model_v4.pdb"
OUTPUT_PDB_NAME = "e7euv6_si.pdb"
FF_NAME = "silicon.ff"

# Paths
input_pdb_path = os.path.join(WORKING_DIR, INPUT_PDB_NAME)
output_pdb_path = os.path.join(WORKING_DIR, OUTPUT_PDB_NAME)
ff_path = os.path.join(WORKING_DIR, FF_NAME)

def get_canonical_key(atoms, func_type):
    """
    Returns a unique key: (Atoms_Tuple, Function_Type)

    CRITICAL: Type 9 improper dihedrals in GROMACS have special symmetry.
    The improper is defined around the central bond (atoms[1]-atoms[2]).
    A-B-C-D and D-C-B-A represent the SAME improper (reversed central bond).

    We must canonicalize type 9 to merge these symmetric forms.
    """
    if not atoms: return tuple()
    t = tuple(atoms)

    # Type 9: Canonicalize considering it's the same improper reversed
    if str(func_type) == "9":
        r = tuple(atoms[::-1])
        canonical_atoms = t if t <= r else r
        return (canonical_atoms, func_type)

    # All other types: canonicalize using symmetry
    r = tuple(atoms[::-1])
    canonical_atoms = t if t <= r else r
    return (canonical_atoms, func_type)

def step1_convert_pdb():
    print(f"--- STEP 1: Processing PDB ---")
    if not os.path.exists(input_pdb_path):
        print("Input file missing.")
        return False
    with open(input_pdb_path, 'r') as f: lines = f.readlines()
    new_lines = []
    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            atom_name = line[12:16]
            if atom_name.strip() == "CA":
                line = line[:12] + " SI " + line[16:]
                if len(line) >= 78: line = line[:76] + "Si" + line[78:]
                else: line = line.strip().ljust(76) + "Si\n"
        new_lines.append(line)
    with open(output_pdb_path, 'w') as f: f.writelines(new_lines)
    print(f"✓ Generated {OUTPUT_PDB_NAME}")
    return True

def step2_patch_ff_global_buffer():
    print(f"\n--- STEP 2: Building Force Field (Global Buffer) ---")
    if os.path.exists(ff_path): shutil.rmtree(ff_path)
    os.makedirs(ff_path)

    # 1. FIND BASE FF
    base_ff_path = None
    if "GMXLIB" in os.environ:
        p = os.path.join(os.environ["GMXLIB"], "amber99sb.ff")
        if os.path.exists(p): base_ff_path = p
    if not base_ff_path:
        for path in ["/usr/share/gromacs/top", "/usr/local/gromacs/share/gromacs/top", "/usr/local/share/gromacs/top"]:
            p = os.path.join(path, "amber99sb.ff")
            if os.path.exists(p): base_ff_path = p; break
    if not base_ff_path:
        print("ERROR: Could not find 'amber99sb.ff'.")
        return False
    print(f"Cloning from: {base_ff_path}")
    for item in os.listdir(base_ff_path):
        shutil.copy2(os.path.join(base_ff_path, item), os.path.join(ff_path, item))

    # 2. PATCH RTP (Strict Column Parsing)
    rtp_file = os.path.join(ff_path, "aminoacids.rtp")
    with open(rtp_file, 'r') as f: lines = f.readlines()
    new_rtp = []
    current_section = None
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("["):
            current_section = stripped.strip("[] ").lower()
            new_rtp.append(line)
            continue
        if current_section == "atoms" and stripped:
            parts = line.split()
            if len(parts) >= 2:
                atom_name = parts[0]
                if atom_name == "CA":
                    line = line.replace(atom_name, "SI", 1)
                    new_parts = line.split()
                    new_parts[1] = "SZ"
                    line = "\t" + "\t".join(new_parts) + "\n"
        elif current_section in ["bonds", "impropers"] and stripped:
            line = re.sub(r'\bCA\b', 'SI', line)
        new_rtp.append(line)
    with open(rtp_file, 'w') as f: f.writelines(new_rtp)

    # 3. PATCH HDB/TDB
    for fname in ["aminoacids.hdb", "aminoacids.n.tdb", "aminoacids.c.tdb"]:
        p = os.path.join(ff_path, fname)
        if os.path.exists(p):
            with open(p, 'r') as f: c = f.read()
            with open(p, 'w') as f: f.write(re.sub(r'\bCA\b', 'SI', c))

    # 4. PATCH NONBONDED
    with open(os.path.join(ff_path, "atomtypes.atp"), 'a') as f:
        f.write("\nSZ      28.08550 ; Silicon Backbone\n")
    nb_file = os.path.join(ff_path, "ffnonbonded.itp")
    with open(nb_file, 'r') as f: lines = f.readlines()
    with open(nb_file, 'w') as f:
        for line in lines:
            f.write(line)
            if re.match(r'^\s*CT\s+6\s+', line):
                f.write(line.replace("CT", "SZ", 1).replace(" 6 ", " 14 ", 1).replace("12.01", "28.08", 1))

    # 5. PATCH BONDED (FIXED FOR TYPE 9 MULTI-LINE)
    print("Generating bonded parameters (Batch Permutation Fix)...")
    bond_file = os.path.join(ff_path, "ffbonded.itp")
    with open(bond_file, 'r') as f: lines = f.readlines()

    # Step 1: Parse original file and collect all entries
    ORIGINAL_ENTRIES = {
        'bondtypes': [],
        'constrainttypes': [],
        'angletypes': [],
        'dihedraltypes': [],
        'implicit_genborn_params': [],
        'pairtypes': []
    }

    valid_sections = list(ORIGINAL_ENTRIES.keys())
    current_section = None

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("["):
            sec = stripped.strip("[] ").lower()
            current_section = sec if sec in valid_sections else "other"
            continue

        if stripped.startswith(";") or not stripped or current_section not in valid_sections:
            continue

        tokens = line.split()
        atom_tokens = []
        func_type = None

        for t in tokens:
            if t[0].isdigit() or t.startswith("-") or "." in t:
                func_type = t
                break
            atom_tokens.append(t)

        if atom_tokens and func_type:
            ORIGINAL_ENTRIES[current_section].append({
                'atoms': atom_tokens,
                'func_type': func_type,
                'tokens': tokens,
                'line': line
            })

    # Step 2: Generate ALL permutations in batch
    ALL_ENTRIES = {sec: [] for sec in valid_sections}

    for sec in valid_sections:
        # First add all originals
        for entry in ORIGINAL_ENTRIES[sec]:
            key = get_canonical_key(entry['atoms'], entry['func_type'])
            ALL_ENTRIES[sec].append((key, entry['line']))

        # Then generate all permutations
        for entry in ORIGINAL_ENTRIES[sec]:
            atoms = entry['atoms']
            tokens = entry['tokens']
            func_type = entry['func_type']

            # Check if any atom is CT
            ct_indices = [i for i, atom in enumerate(atoms) if atom == "CT"]

            if ct_indices:
                combinations = list(itertools.product(["CT", "SZ"], repeat=len(ct_indices)))
                combinations.pop(0)  # Remove all-CT (already added)

                for combo in combinations:
                    new_tokens = list(tokens)
                    for i, replacement in enumerate(combo):
                        new_tokens[ct_indices[i]] = replacement

                    new_line = "\t".join(new_tokens) + "\n"
                    new_atoms = new_tokens[:len(atoms)]
                    new_key = get_canonical_key(new_atoms, func_type)

                    ALL_ENTRIES[sec].append((new_key, new_line))

    # Debug: Check for duplicates in ALL_ENTRIES before deduplication
    for sec in valid_sections:
        if sec == 'dihedraltypes':
            entries_dict = defaultdict(list)
            for key, line in ALL_ENTRIES[sec]:
                line_norm = '\t'.join(line.split())
                entries_dict[(key, line_norm)].append(line)

            multi = [(k, len(v)) for k, v in entries_dict.items() if len(v) > 1]
            if multi:
                print(f"\n[{sec}] Found {len(multi)} duplicate entries in ALL_ENTRIES before dedup:")
                for (key, line_norm), count in multi[:5]:  # Show first 5
                    atoms = ' '.join(str(x) for x in key[0])
                    print(f"  {count}x: {atoms} (type {key[1]})")

    # Step 3: Deduplicate and group by key AND parameters
    FINAL_STORE = {}
    FINAL_ORDER = {}

    for sec in valid_sections:
        FINAL_STORE[sec] = defaultdict(list)
        FINAL_ORDER[sec] = []
        seen = set()  # Track (key, parameters) to keep one line per parameter set
        keys_seen = set()
        duplicate_count = 0

        for key, line in ALL_ENTRIES[sec]:
            tokens = line.split()

            # Extract parameters (everything after the atoms and func_type)
            # Find where func_type is (first numeric token)
            func_idx = None
            for i, t in enumerate(tokens):
                if t[0].isdigit() or t.startswith('-'):
                    func_idx = i
                    break

            if func_idx is None:
                continue

            # Parameters = func_type + everything after (force constants, multiplicities, etc.)
            # Exclude comments (tokens starting with ;)
            params = []
            for t in tokens[func_idx:]:
                if t.startswith(';'):
                    break
                params.append(t)

            params_tuple = tuple(params)
            entry_sig = (key, params_tuple)

            if entry_sig in seen:
                duplicate_count += 1
                continue

            if key not in keys_seen:
                FINAL_ORDER[sec].append(key)
                keys_seen.add(key)

            FINAL_STORE[sec][key].append(line)
            seen.add(entry_sig)

        if duplicate_count > 0:
            print(f"  [{sec}]: Filtered out {duplicate_count} duplicate parameter sets")

    # Step 4: Write file with diagnostics
    with open(bond_file, 'w') as f:
        f.write("; Generated by Silicon-Patcher 19.0 (Diagnostic)\n")

        write_order = ["bondtypes", "constrainttypes", "angletypes", "dihedraltypes",
                       "implicit_genborn_params", "pairtypes"]

        for sec in write_order:
            if not FINAL_STORE[sec]: continue

            f.write(f"\n[ {sec} ]\n")

            # Track what we write to detect GROMACS' complaint
            written_keys = {}  # key -> line number where first written
            current_line = 0

            for key in FINAL_ORDER[sec]:
                lines_for_key = FINAL_STORE[sec][key]

                # Check if this key was already written (shouldn't happen)
                if key in written_keys:
                    print(f"WARNING: Duplicate key detected in {sec}!")
                    print(f"  Key: {key}")
                    print(f"  First block at line: {written_keys[key]}")
                    print(f"  Second block at line: {current_line}")
                    print(f"  Lines in second block: {len(lines_for_key)}")

                written_keys[key] = current_line

                for line in lines_for_key:
                    f.write(line)
                    current_line += 1

    print(f"✓ Force Field patched.")
    for sec in write_order:
        if FINAL_STORE[sec]:
            total_keys = len(FINAL_ORDER[sec])
            total_lines = sum(len(FINAL_STORE[sec][k]) for k in FINAL_ORDER[sec])
            multi_line = sum(1 for k in FINAL_ORDER[sec] if len(FINAL_STORE[sec][k]) > 1)
            print(f"  [{sec}]: {total_keys} unique keys, {total_lines} lines, {multi_line} multi-line blocks")

    # CRITICAL: Dump lines around error to debug
    print("\n--- DEBUG: Checking dihedraltypes section ---")
    with open(bond_file, 'r') as f:
        all_lines = f.readlines()
        in_dihedrals = False
        line_num = 0
        dihedral_lines = []

        for line in all_lines:
            line_num += 1
            if '[ dihedraltypes ]' in line:
                in_dihedrals = True
                continue
            if in_dihedrals:
                if line.strip().startswith('['):
                    break
                if line.strip() and not line.strip().startswith(';'):
                    dihedral_lines.append((line_num, line.strip()))

        # Show lines 770-785 (around error line 778-780)
        print(f"Total dihedral lines: {len(dihedral_lines)}")
        print("\nLines around 770-785 (FULL CONTENT):")
        for num, content in dihedral_lines:
            if 770 <= num <= 785:
                print(f"  Line {num}: {content}")

        # Check for exact duplicates
        print("\n--- Checking for exact duplicate lines ---")
        seen_lines = {}
        for num, content in dihedral_lines:
            if content in seen_lines:
                print(f"  DUPLICATE: Line {num} is identical to line {seen_lines[content]}")
                print(f"    Content: {content}")
            else:
                seen_lines[content] = num

    print("✓ Force Field patched (Type 9 Multi-line Support).")
    return True

def step3_run_gromacs_pipeline():
    print(f"\n--- STEP 3: Running GROMACS Pipeline ---")
    env = os.environ.copy()
    env["GMXLIB"] = WORKING_DIR

    for f in ["processed.gro", "topol.top", "box.gro", "solvated.gro", "em.tpr", "em.gro", "em.log", "em.edr"]:
        if os.path.exists(f): os.remove(f)

    # 1. PDB2GMX
    print("1. Generating Topology...")
    subprocess.run(["gmx", "pdb2gmx", "-f", output_pdb_path, "-o", "processed.gro", "-p", "topol.top", "-ff", "silicon", "-water", "tip3p", "-ignh"], env=env, check=True)

    # 2. BOX
    print("2. Defining Box...")
    subprocess.run(["gmx", "editconf", "-f", "processed.gro", "-o", "box.gro", "-c", "-d", "1.0", "-bt", "cubic"], env=env, check=True)

    # 3. MINIM
    print("3. Preparing Minimization...")
    with open("minim.mdp", "w") as f: f.write("integrator=steep\nnsteps=1000\nemtol=1000\nemstep=0.01\n")
    subprocess.run(["gmx", "grompp", "-f", "minim.mdp", "-c", "box.gro", "-p", "topol.top", "-o", "em.tpr", "-maxwarn", "5"], env=env, check=True)

    # 4. RUN
    print("4. Running Minimization...")
    subprocess.run(["gmx", "mdrun", "-v", "-deffnm", "em"], env=env, check=True)
    print("\nSUCCESS: Minimization complete! 'em.gro' is ready.")

if __name__ == "__main__":
    if step1_convert_pdb():
        if step2_patch_ff_global_buffer():
            try:
                step3_run_gromacs_pipeline()
            except subprocess.CalledProcessError:
                print("\nFAILURE: Pipeline stopped.")

--- STEP 1: Processing PDB ---
✓ Generated e7euv6_si.pdb

--- STEP 2: Building Force Field (Global Buffer) ---
Cloning from: /usr/local/gromacs/share/gromacs/top/amber99sb.ff
Generating bonded parameters (Batch Permutation Fix)...
  [bondtypes]: Filtered out 1 duplicate parameter sets
  [angletypes]: Filtered out 13 duplicate parameter sets
  [dihedraltypes]: Filtered out 31 duplicate parameter sets
✓ Force Field patched.
  [bondtypes]: 120 unique keys, 120 lines, 0 multi-line blocks
  [constrainttypes]: 13 unique keys, 13 lines, 0 multi-line blocks
  [angletypes]: 368 unique keys, 368 lines, 0 multi-line blocks
  [dihedraltypes]: 284 unique keys, 385 lines, 77 multi-line blocks

--- DEBUG: Checking dihedraltypes section ---
Total dihedral lines: 385

Lines around 770-785 (FULL CONTENT):
  Line 770: H1	SZ	C	O	9	0.0	3.34720	1	;	Junmei	et	al,	1999
  Line 771: H1	SZ	C	O	9	180.0	0.33472	3	;	Junmei	et	al,	1999
  Line 772: HC	SZ	C	O	9	0.0	3.34720	1	;	Junmei	et	al,	1999
  Line 773: HC	SZ	C	O	

In [ ]:
%cd /content/drive/MyDrive/MD/Silica/

!gmx pdb2gmx -f e7euv6_si.pdb -o processed.gro -p topol.top -i posre.itp -ff silicon -water tip3p

# 4. Define the Simulation Box
!gmx editconf -f processed.gro -o box.gro -c -d 1.0 -bt cubic

# 5. Solvate the Box (Add Water)
!gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top

# 6. Add Ions (Neutralize system)
# First generate a dummy tpr
!gmx grompp -f minim.mdp -c solvated.gro -p topol.top -o ions.tpr
# Replace SOL with ions
!echo "SOL" | gmx genion -s ions.tpr -o solvated_ions.gro -p topol.top -pname NA -nname CL -neutral

# 7. Energy Minimization
# This is CRITICAL for Silicon integration to relax the bonds
!gmx grompp -f minim.mdp -c solvated_ions.gro -p topol.top -o em.tpr
!gmx mdrun -v -deffnm em

/content/drive/MyDrive/MD/Silica
                     :-) GROMACS - gmx pdb2gmx, 2024.4 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /content/drive/MyDrive/MD/Silica
Command line:
  gmx pdb2gmx -f e7euv6_si.pdb -o processed.gro -p topol.top -i posre.itp -ff silicon -water tip3p

Using the Silicon force field in directory ./silicon.ff

going to rename ./silicon.ff/aminoacids.r2b
Opening force field file ./silicon.ff/aminoacids.r2b

going to rename ./silicon.ff/dna.r2b
Opening force field file ./silicon.ff/dna.r2b

going to rename ./silicon.ff/rna.r2b
Opening force field file ./silicon.ff/rna.r2b
Reading e7euv6_si.pdb...
Read 'AMINO ACID TRANSPORTER', 858 atoms

Analyzing pdb file
Splitting chemical chains based on TER records or chain id changing.

There are 1 chains and 0 blocks of water and 106 residues with 858 atoms

  chain  #res #atoms

  1 'A'   106    858  

All occupancies are one
All occupancies are one
Opening force field file .

In [ ]:
import os
import subprocess

# --- CONFIGURATION ---
WORKING_DIR = "/content/drive/MyDrive/MD/Silica/"
FF_NAME = "silicon.ff"
OUTPUT_PDB_NAME = "e7euv6_si.pdb"

def step4_run_nvt():
    print(f"\n--- STEP 4: NVT Equilibration (Heating to 300K) ---")

    # 1. Setup Environment (Ensure GROMACS finds our local silicon.ff)
    env = os.environ.copy()
    env["GMXLIB"] = WORKING_DIR

    # 2. Write NVT Parameters (nvt.mdp)
    # We use V-rescale thermostat to heat from 0K to 300K
    mdp_content = """
title                   = Protein NVT equilibration
define                  = -DPOSRES  ; Position restrain the protein (keep it stable while water relaxes)

; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 50000     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
comm-mode               = Linear    ; Remove center of mass translation

; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps

; Bond parameters
continuation            = no        ; first dynamics run
constraint_algorithm    = lincs     ; holonomic constraints
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy

; Nonbonded settings
cutoff-scheme           = Verlet    ; Buffered neighbor searching
ns_type                 = grid      ; search neighboring grid cells
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet scheme
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme

; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT

; Temperature coupling is on
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = Protein Non-Protein   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K

; Pressure coupling is off
pcoupl                  = no        ; no pressure coupling in NVT

; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC

; Velocity generation
gen_vel                 = yes       ; assign velocities from Maxwell distribution
gen_temp                = 300       ; temperature for Maxwell distribution
gen_seed                = -1        ; generate a random seed
    """

    with open("nvt.mdp", "w") as f:
        f.write(mdp_content)
    print("✓ Created nvt.mdp")

    # 3. Prepare Run (grompp)
    # Input: em.gro (minimized structure), topol.top (topology)
    # Output: nvt.tpr (binary run input)
    print("Running grompp for NVT...")
    cmd_grompp = [
        "gmx", "grompp",
        "-f", "nvt.mdp",
        "-c", "em.gro",
        "-r", "em.gro",      # Restraint positions (same as input)
        "-p", "topol.top",
        "-o", "nvt.tpr",
        "-maxwarn", "5"      # Silicon parameters might trigger warnings, usually safe to ignore if minimized
    ]
    subprocess.run(cmd_grompp, env=env, check=True)

    # 4. Run MD (mdrun)
    print("Running NVT Simulation (this may take a few minutes)...")
    cmd_mdrun = [
        "gmx", "mdrun",
        "-v",                # Verbose
        "-deffnm", "nvt"     # Output files will be nvt.gro, nvt.edr, etc.
    ]
    subprocess.run(cmd_mdrun, env=env, check=True)

    print("\nSUCCESS: NVT Equilibration complete!")
    print("Output file: nvt.gro (Heated, equilibrated structure)")

if __name__ == "__main__":
    try:
        step4_run_nvt()
    except subprocess.CalledProcessError:
        print("\nFAILURE: NVT step failed. Check the output above.")


--- STEP 4: NVT Equilibration (Heating to 300K) ---
✓ Created nvt.mdp
Running grompp for NVT...
Running NVT Simulation (this may take a few minutes)...

SUCCESS: NVT Equilibration complete!
Output file: nvt.gro (Heated, equilibrated structure)


In [ ]:
%cd /content/drive/MyDrive/MD/Silica/
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 5
!gmx mdrun -v -deffnm npt

In [ ]:
%cd /content/drive/MyDrive/MD/Silica/
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 5
!gmx mdrun -v -deffnm md